# 🧪 EE-559 Experiment Plan: English Religious Hate Speech Detection

This document outlines the fine-tuning strategy for binary classification (Religious Hate vs. Not Hate) using several transformer architectures. All experiments are conducted in **English only**.

---

## 🎯 Objective

Develop a performant and efficient text classifier that identifies religious hate speech in user-generated content.

---

## 📚 Dataset

| Split  | File                   | Size      |
|--------|------------------------|-----------|
| Train  | `train_balanced.csv`   | Balanced  |
| Val    | `val_balanced.csv`     | Balanced  |
| Test   | `test_balanced.csv`    | Balanced  |
| Target | Binary: Hate (1), Not Hate (0) |

---

## 🧠 Models to Fine-Tune

| Model Name                      | Params | Notes                                     |
|--------------------------------|--------|--------------------------------------------|
| `bert-tiny`                    | ~4M    | 🚀 Fastest baseline                        |
| `distilbert-base-uncased` ✅   | ~66M   | ✅ Already trained                         |
| `bert-base-uncased`            | ~110M  | 🧠 Classic BERT                            |
| `roberta-base`                 | ~125M  | Strong encoder, robust pretraining         |
| `albert-base-v2`               | ~12M   | Parameter-efficient alternative            |
| `electra-small-discriminator` | ~14M   | Fast, trained with replaced token detection |
| `google/bert_uncased_L-4_H-256_A-4` | ~10M | Compact & fast, good for ablation         |

---

## ⚙️ Training Settings

| Parameter        | Values to Try                             |
|------------------|--------------------------------------------|
| **Epochs**        | `3`, `5`                                    |
| **Batch Size**    | `16`, `32`                                  |
| **Max Length**    | `128`, `256`                                |
| **Learning Rate** | `5e-5`, `3e-5`, `1e-5`                      |
| **Optimizer**     | `AdamW`, `SGD`                              |
| **Scheduler**     | `None`, `linear`, `cosine`                 |
| **Dropout**       | Default (may test 0.1, 0.3 on selected models) |

---

## 📏 Evaluation Metrics

- **Primary:** F1 Score (macro)
- **Secondary:** Accuracy, Precision, Recall
- Optional: Confusion Matrix per model

---

## 🧪 Run Matrix (Initial Settings)

| Model                       | LR    | Max Len | Epochs | Batch | Notes                    |
|----------------------------|-------|---------|--------|--------|---------------------------|
| `bert-tiny`                | 5e-5  | 128     | 3      | 32     | Very fast baseline        |
| `distilbert-base` ✅       | 3e-5  | 256     | 5      | 16     | Already trained! ✅        |
| `bert-base-uncased`        | 3e-5  | 256     | 3      | 16     | Standard setup             |
| `roberta-base`             | 1e-5  | 256     | 3      | 16     | Strong encoding model      |
| `albert-base-v2`           | 5e-5  | 256     | 5      | 32     | Efficient architecture     |
| `electra-small-discriminator` | 3e-5 | 128  | 3      | 32     | Unique pretraining objective |
| `bert_uncased_L4_H256_A-4` | 3e-5  | 128     | 3      | 32     | Lightweight BERT variant   |

🔁 Optional: Try second run with:
- `SGD` + `cosine scheduler` for `bert-base`, `roberta-base`
- `Max length 128` for fast ablation on `albert-base`, `electra-small`

---

## 🏆 Model Comparison Plan

- Compare all on **validation set**
- Best models evaluated on **test set**
- Select final model for Gradio app deployment

---

## 💻 Deployment

Final model will be:
- Integrated into `app.py`
- Evaluated with test results (F1/accuracy)
- Documented clearly in report

---


## ✅ Summary

- 7 models trained, 1 already done
- Key settings tested (lr, epochs, schedulers)
- Efficient mix of fast & strong models
- Focused, non-exhaustive grid search

---


## 💻 Install required packages if not already installed

In [1]:
! pip install torch transformers pandas scikit-learn tqdm


import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    get_scheduler
)
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm

  Using cached torch-2.2.2-cp311-none-macosx_10_9_x86_64.whl.metadata (25 kB)
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.3.2-py3-none-any.whl.metadata (11 kB)
  Using cached PyYAML-6.0.2-cp311-cp311-macosx_10_9_x86_64.whl.metadata (2.1 kB)
  Using cached regex-2024.11.6-cp311-cp311-macosx_10_9_x86_64.whl.metadata (40 kB)
  Using cached safetensors-0.5.3-cp38-abi3-macosx_10_12_x86_64.whl.metadata (3.8 kB)
  Using cached MarkupSafe-3.0.2-cp311-cp311-macosx_10_9_universal2.whl.metadata (4.0 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 10.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 10.2 MB/s eta 0:00:00 0:00:01
Using cached fsspec-2025.3.2-py3-none-any.whl (194 kB)
Using cached PyYAML-6.0.2-cp311-cp311-macosx_10_9_x86_64.whl (184 kB)
Using cached regex-2024.11.6


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/gs/.pyenv/versions/3.11.0/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/gs/.pyenv/versions/3.11.0/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/gs/.pyenv/versions/3.11.0/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_lo

## ✅ Check if CUDA is available

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")


🖥️ Using device: cpu


## 🧪 Check versions of key packages

In [4]:
import sys
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
import transformers
print(f"Transformers version: {transformers.__version__}")

Python version: 3.11.0 (main, Mar 12 2025, 14:29:35) [Clang 16.0.0 (clang-1600.0.26.6)]
PyTorch version: 2.2.2
Transformers version: 4.51.3
